In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob

from spikeinterface_pipeline import BapunSessionConfig, CurationConfig, resolve_session_paths, load_bapun_recording, load_phy_sorting, load_spykingcircus_sorting, run_phy_curation_pipeline, compute_qm_labels


Automatic pdb calling has been turned OFF


/home/halechr/FastData/Bapun/RatS/NeuroPy/neuropy/utils/mixins/time_slicing.py:404: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("time_point_event")


In [9]:
## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day1Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# # n_channels: int = 200
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day1Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux

# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day4Openfield') # Greatlakes
# # basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# # basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day4Openfield'
# excluded_data_datetimes = []


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day5TwoNovel') # Linux
basedir = Path('/media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day5TwoNovel')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day5TwoNovel") # Apogee WSL
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day5TwoNovel'
basename: str = 'RatS-Day5TwoNovel-2020-12-04_07-55-09'
excluded_data_datetimes = []

In [10]:
session = BapunSessionConfig(basedir=basedir, basename=basename, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate, eeg_sampling_rate=eeg_sampling_rate, excluded_data_datetimes=excluded_data_datetimes)
paths = resolve_session_paths(session)
xml_path = paths.xml_path
concatenated_dat_file = paths.concatenated_dat_file
print(f'xml_path: {xml_path}')
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."


xml_path: /media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel/RatS-Day5TwoNovel-2020-12-04_07-55-09.xml
concatenated_dat_file: /media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel/spyk-circ/RatS-Day5TwoNovel-2020-12-04_07-55-09.dat


In [11]:
spiking_circus_loaded = load_spykingcircus_sorting(paths)
spiking_circus_loaded


AssertionError: Not a valid spyking circus folder

In [12]:
phy_gui_dir = paths.phy_gui_dir
assert phy_gui_dir.exists(), f"phy_gui_dir: {phy_gui_dir} does not exist!"


AssertionError: phy_gui_dir: /media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel/spyk-circ/RatS-Day5TwoNovel-2020-12-04_07-55-09/RatS-Day5TwoNovel-2020-12-04_07-55-09-merged.GUI does not exist!

In [13]:
sorting = load_phy_sorting(paths)
sorting


FileNotFoundError: phy_gui_dir does not exist: /media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel/spyk-circ/RatS-Day5TwoNovel-2020-12-04_07-55-09/RatS-Day5TwoNovel-2020-12-04_07-55-09-merged.GUI

In [14]:
recording, recording_filtered = load_bapun_recording(session, paths)
recording


ChannelSliceRecording: 176 channels - 30.0kHz - 1 segments - 1,269,174,528 samples 
                       42,305.82s (11.75 hours) - int16 dtype - 416.07 GiB

In [15]:
recording_filtered


BandpassFilterRecording: 176 channels - 30.0kHz - 1 segments - 1,269,174,528 samples 
                         42,305.82s (11.75 hours) - int16 dtype - 416.07 GiB

In [ ]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')


In [ ]:

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [ ]:
# run Tridesclous
TDC_Output_Path: Path = sorting_outputs_folder.joinpath("folder_TDC")
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording_filtered, folder=TDC_Output_Path)
sorting_TDC

In [ ]:
TDC_Final_Sorting_Output_Path: Path = TDC_Output_Path.joinpath('tridescloud_sorting_output')
sorting_TDC.save(folder=TDC_Final_Sorting_Output_Path)


In [ ]:
sorting_TDC.sorting_info

In [ ]:
# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder=sorting_outputs_folder.joinpath("folder_KS4"))

In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"))
sorting_SC2

In [ ]:
sorting_SC2


# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=recording_filtered)
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])


In [ ]:
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

In [30]:
all_metric_names = list(sorting_analyzer.get_extension('quality_metrics').get_data().keys()) + list(sorting_analyzer.get_extension('template_metrics').get_data().keys())
print(set(model.feature_names_in_).issubset(set(all_metric_names)))

NameError: name 'model' is not defined

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

In [ ]:
labels = sc.model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trusted = ['numpy.dtype']
)

print(labels)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)

# Trained Models

In [ ]:
from spikeinterface.curation import model_based_label_units

labels_and_probabilities = model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trust_model = True
)

In [ ]:
GOOD_UNIT_STRATEGY: str = "sua_high_conf"
PROB_THRESHOLD_HIGH: float = 0.8
curation = CurationConfig(strategy="sua_high_conf", prob_high=PROB_THRESHOLD_HIGH, analyzer_overwrite="always", require_cluster_info=False)


In [ ]:
result = run_phy_curation_pipeline(session, curation, patch_pandas_compat=True)
sorting = result.sorting
sorting_analyzer = result.sorting_analyzer
all_labels = result.all_labels
comparison = result.comparison
good_units = result.good_units
sorting_curated = result.sorting_curated
print(all_labels["prediction"].value_counts())
print(comparison.groupby(["phy_label", "prediction"]).size())


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
sorting

In [27]:
metrics_df = sorting_analyzer.get_extension("quality_metrics").get_data()
template_df = sorting_analyzer.get_extension("template_metrics").get_data()

out_dir = sorting_analyzer_folder / "exports"
out_dir.mkdir(exist_ok=True)
metrics_df.to_csv(out_dir / "quality_metrics.csv")
template_df.to_csv(out_dir / "template_metrics.csv")

In [28]:
metrics_df

,presence_ratio,snr,amplitude_cutoff,isolation_distance,l_ratio,d_prime
0,0.937500,7.844408,0.000000,15.847837,1.751640,1.902061
1,0.991065,2.938588,0.000000,NaN,NaN,NaN
3,0.989977,2.473773,0.000000,24.148419,0.577404,2.258721
4,0.990545,2.974413,0.000000,23.989873,0.573163,2.199788
5,0.754586,11.318492,0.000000,21.134439,1.524469,2.475873
8,0.564297,15.982475,0.000000,35.985235,0.281415,3.733157
11,0.942748,5.807396,0.000000,34.678190,0.661975,2.121639
14,0.975180,3.926572,0.000000,39.228007,0.498313,2.422435
24,0.992388,3.234437,0.000000,21.975400,0.894074,2.127699
26,0.993334,2.939547,0.000000,31.874253,0.633763,2.502093


In [29]:
template_df

,peak_to_trough_duration,trough_half_width,peak_half_width,repolarization_slope,recovery_slope,num_positive_peaks,num_negative_peaks,main_to_next_extremum_duration,peak_before_to_trough_ratio,peak_after_to_trough_ratio,peak_before_to_peak_after_ratio,main_peak_to_trough_ratio,trough_width,peak_before_width,peak_after_width,waveform_baseline_flatness,velocity_above,velocity_below,exp_decay,spread
0,0.000630,0.000213,0.000950,3.516864e+05,-11618.440627,0,1,0.000630,0.123113,0.192427,0.639791,0.192427,0.000235,NaN,0.000951,0.020786,NaN,-750.000000,0.021727,120.0
1,0.000987,0.000417,0.000880,8.407578e+04,-18716.231594,1,1,0.000987,0.247462,0.384747,0.643182,0.384747,0.000492,NaN,0.000903,0.021324,NaN,NaN,0.030283,90.0
3,0.000997,0.000327,0.000893,8.402904e+04,-18153.470322,1,1,0.000997,0.162472,0.340225,0.477541,0.340225,0.000379,NaN,0.000903,0.051820,NaN,655.681881,0.022196,105.0
4,0.000993,0.000300,0.000973,1.149195e+05,-15278.459904,1,1,0.000993,0.118068,0.306317,0.385446,0.306317,0.000329,NaN,0.000974,0.068679,NaN,NaN,0.027313,105.0
5,0.000920,0.000200,0.000870,6.435659e+05,-34065.318313,0,1,0.000920,0.126475,0.163347,0.774270,0.163347,0.000219,NaN,0.000944,0.020984,NaN,NaN,0.026505,135.0
8,0.000293,0.000160,0.000473,1.628396e+06,-37288.880301,0,1,0.000293,0.080953,0.178400,0.453771,0.178400,0.000170,NaN,0.000629,0.011283,NaN,NaN,0.026888,135.0
11,0.000513,0.000230,0.000743,3.680703e+05,-18385.001344,0,1,0.000513,0.145386,0.221611,0.656041,0.221611,0.000253,NaN,0.000784,0.022850,961.666788,NaN,0.031717,135.0
14,0.000673,0.000260,0.000810,2.051705e+05,-15019.040934,0,1,0.000673,0.166096,0.263525,0.630284,0.263525,0.000295,NaN,0.000850,0.029208,NaN,-1565.217391,0.023109,135.0
24,0.000910,0.000317,0.000773,1.151087e+05,-21652.518818,1,1,0.000910,0.182659,0.317782,0.574794,0.317782,0.000370,NaN,0.000842,0.047342,1125.000000,609.115641,0.028180,120.0
26,0.001157,0.000343,0.000780,1.025369e+05,-28420.288291,1,1,0.001157,0.227469,0.420839,0.540514,0.420839,0.000412,NaN,0.000792,0.060106,NaN,NaN,0.021097,120.0


## Write out labels to Phy

In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [18]:
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s") 
sorting_analyzer_folder: Path = Path('/media/halechr/BETAMAX1/Data/Bapun/RatS/Day5TwoNovel/SORTING/folder_SC2_sorting_analyzer')
sorting_analyzer = si.load_sorting_analyzer(sorting_analyzer_folder.as_posix())
sorting_analyzer


SortingAnalyzer: 176 channels - 38 units - 1 segments - binary_folder - sparse - has recording
Loaded 3 extensions: random_spikes, waveforms, templates

In [ ]:

# sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])

In [ ]:
# Define the extensions you want, along with their parameters
extensions_to_run = {
    "random_spikes": {"max_spikes_per_unit": 500},
    "waveforms": {"ms_before": 1.0, "ms_after": 2.0},
    "templates": {"operators": ["average", "median"]},
    "noise_levels": {},
    "principal_components": {"n_components": 3, "mode": "by_channel_local"},
    'template_metrics': {'include_multi_channel_metrics': True},
    "quality_metrics": dict(    metric_names=["presence_ratio", "snr", "amplitude_cutoff", "mahalanobis", "d_prime", 'amplitude_median', 'num_spikes', 'rp_contamination', 'drift_ptp'],
                                metric_params={
                                    "presence_ratio": {"bin_duration_s": 2.0},
                                }),
}

extensions_to_run = extensions_to_run | {k:{} for k in ['spike_locations','spike_amplitudes','correlograms']}
extensions_to_run

# Iterate through and compute only if the extension is missing
# for ext_name, ext_params in extensions_to_run.items():
#     if not sorting_analyzer.has_extension(ext_name):
#         print(f"Computing missing extension: {ext_name}")
#         sorting_analyzer.compute(ext_name, **ext_params)
#     else:
#         print(f"Extension '{ext_name}' already exists. Skipping.")


active_extensions_to_run = {ext_name:ext_params for ext_name, ext_params in extensions_to_run.items() if (not sorting_analyzer.has_extension(ext_name))}
active_extensions_to_run


{'noise_levels': {},
 'principal_components': {'n_components': 3, 'mode': 'by_channel_local'},
 'template_metrics': {'include_multi_channel_metrics': True},
 'quality_metrics': {'metric_names': ['presence_ratio',
   'snr',
   'amplitude_cutoff',
   'mahalanobis',
   'd_prime'],
  'metric_params': {'presence_ratio': {'bin_duration_s': 2.0}}},
 'spike_locations': {},
 'spike_amplitudes': {},
 'correlograms': {}}

In [23]:
si.set_global_job_kwargs(n_jobs=7, chunk_duration="1s", progress_bar=True)

sorting_analyzer.compute(active_extensions_to_run)

# If you computed new extensions and your analyzer is memory-backed, 
# remember to save it afterwards:
# analyzer.save_as(format="zarr", folder="/path/to/new_analyzer.zarr")

noise_level (workers: 7 processes fork):   0%|          | 0/20 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/38 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/38 [00:00<?, ?it/s]

/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/metrics/template/template_metrics.py:304: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/postprocessing/correlograms.py:737: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  numba.set_num_threads(num_threads)


Compute : spike_amplitudes + spike_locations (workers: 7 processes fork):   0%|          | 0/42306 [00:00<?, ?…

/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = 

In [35]:
sorting_analyzer.compute(({"quality_metrics": dict(    metric_names=["presence_ratio", "snr", "amplitude_cutoff", "mahalanobis", "d_prime", 'amplitude_median', 'num_spikes', 'rp_contamination', 'drift_ptp'],
                                metric_params={
                                    "presence_ratio": {"bin_duration_s": 2.0},
                                }),
}))

ValueError: Metric rp_contamination not in available metrics ['num_spikes', 'firing_rate', 'presence_ratio', 'snr', 'isi_violation', 'rp_violation', 'sliding_rp_violation', 'synchrony', 'firing_range', 'amplitude_cv', 'amplitude_cutoff', 'noise_cutoff', 'amplitude_median', 'drift', 'sd_ratio', 'mahalanobis', 'd_prime', 'nearest_neighbor', 'silhouette', 'nn_advanced']

In [25]:
sorting_analyzer_export_path: Path = sorting_analyzer_folder.joinpath(f"2026-06-05_sorting_analyzer.zarr")
assert not sorting_analyzer_export_path.exists()

In [26]:
sorting_analyzer.save_as(format="zarr", folder=sorting_analyzer_export_path)

SortingAnalyzer: 176 channels - 38 units - 1 segments - zarr - sparse - has recording
Loaded 10 extensions: random_spikes, waveforms, templates, noise_levels, principal_components, template_metrics, correlograms, spike_amplitudes, spike_locations, quality_metrics

# 2026-06-05 6:14am - Compute

In [37]:
qm = sorting_analyzer.get_extension("quality_metrics").get_data()
print(qm.isna().sum())   # <-- THE key check: which metrics are NaN for every unit?
print(qm.describe())     # <-- real distribution of snr / isi / contamination

presence_ratio        0
snr                   0
amplitude_cutoff      0
isolation_distance    7
l_ratio               7
d_prime               7
dtype: int64
       presence_ratio        snr  amplitude_cutoff  isolation_distance  \
count       38.000000  38.000000         38.000000           31.000000   
mean         0.965658   5.508561          0.008489         5951.520701   
std          0.086319   3.062202          0.033946        32610.411447   
min          0.564297   2.335782          0.000000           15.847837   
25%          0.990072   3.243907          0.000000           32.148470   
50%          0.993168   4.154378          0.000000           36.978790   
75%          0.998133   6.868417          0.000000           54.808123   
max          0.999291  15.982475          0.183656       181656.789784   

         l_ratio    d_prime  
count  31.000000  31.000000  
mean    0.697311   2.376610  
std     0.420568   0.574757  
min     0.023391   1.460829  
25%     0.398620   2.12269

In [34]:
from spikeinterface.curation import threshold_metrics_label_units

labels = threshold_metrics_label_units(metrics_df, thresholds={"isi_violations_ratio": {"less": 0.5}, "rp_contamination": {"less": 0.3}, "presence_ratio": {"greater": 0.8}}, pass_label="good", fail_label="mua", column_name="auto_label")
print(labels["auto_label"].value_counts())

ValueError: Metric(s) ['isi_violations_ratio', 'rp_contamination'] specified in thresholds are not present in the quality metrics DataFrame. Available metrics are: ['presence_ratio', 'snr', 'amplitude_cutoff', 'isolation_distance', 'l_ratio', 'd_prime']

In [33]:
from spikeinterface.curation import bombcell_label_units, bombcell_get_default_thresholds

labels = bombcell_label_units(sorting_analyzer=sorting_analyzer, thresholds=bombcell_get_default_thresholds()) # , thresholds=bombcell_get_default_thresholds()
labels

ValueError: Metric(s) ['amplitude_median', 'num_spikes', 'rp_contamination', 'drift_ptp'] specified in thresholds are not present in the quality metrics DataFrame. Available metrics are: ['presence_ratio', 'snr', 'amplitude_cutoff', 'isolation_distance', 'l_ratio', 'd_prime', 'peak_to_trough_duration', 'trough_half_width', 'peak_half_width', 'repolarization_slope', 'recovery_slope', 'num_positive_peaks', 'num_negative_peaks', 'main_to_next_extremum_duration', 'peak_before_to_trough_ratio', 'peak_after_to_trough_ratio', 'peak_before_to_peak_after_ratio', 'main_peak_to_trough_ratio', 'trough_width', 'peak_before_width', 'peak_after_width', 'waveform_baseline_flatness', 'velocity_above', 'velocity_below', 'exp_decay', 'spread']

In [ ]:
import bombcell as bc


# Replace with your kilosort directory
ks_dir = "toy_data"

# Set bombcell's output directory
save_path = Path(ks_dir) / "bombcell"

print(f"Using kilosort directory: {ks_dir}")


